# 01. Sales Baseline, Schema Audit & Time-Series Decomposition

Audits daily revenue, hourly demand curves (clamped to operating hours `06:00–21:00`), and 4-way SKU mix.
- **Default `DATA_MODE="sample"`**: runs out-of-the-box using synthetic CSVs in `data/sample/`.
- **Local `DATA_MODE="raw"`**: reads local SQLite DBs in `data/raw/` via read-only allowlisted loaders (`src.db_loader`).

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("BGA_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/bga-matplotlib")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from src.config import load_config

DATA_MODE = os.environ.get("BGA_DATA_MODE", "sample").strip().lower()
cfg = load_config(repo_root=REPO_ROOT)
print(f"DATA_MODE={DATA_MODE} | branch={cfg.branch_id} | coords=({cfg.lat}, {cfg.lon}) verified={cfg.coords_verified}")

In [ ]:
if DATA_MODE == "raw":
    from src.db_loader import inspect_schema, load_daily_sales, load_hourly_sales, load_product_mix
    schema = inspect_schema(cfg.master_sales_db_path, db_kind="master_sales")
    print("Allowlisted tables:", sorted(schema.keys()))
    daily_df = load_daily_sales(config=cfg)
    hourly_df = load_hourly_sales(config=cfg, complete_grid=True)
    mix_df = load_product_mix(config=cfg)
else:
    daily_df = pd.read_csv(cfg.sample_dir / "daily_sales.csv", parse_dates=["date"])
    hourly_df = pd.read_csv(cfg.sample_dir / "hourly_sales.csv", parse_dates=["date"])
    mix_df = pd.read_csv(cfg.sample_dir / "product_mix.csv", parse_dates=["date"])

print(f"Daily rows={len(daily_df)} ({daily_df['date'].min().date()} .. {daily_df['date'].max().date()})")
print(f"Hourly rows={len(hourly_df)} | hours={sorted(hourly_df['hour_int'].unique().tolist())}")
print(f"Product mix rows={len(mix_df)} | categories={mix_df['category'].value_counts().to_dict()}")

In [ ]:
daily_df = daily_df.sort_values("date").copy()
daily_df["dow"] = daily_df["date"].dt.day_name()
daily_df["rolling_14d"] = daily_df["revenue"].rolling(14, min_periods=3).mean()

dow_summary = (
    daily_df.groupby(daily_df["date"].dt.day_of_week, as_index=False)
    .agg(dow=("dow", "first"), mean_revenue=("revenue", "mean"), mean_tickets=("tickets", "mean"), avg_ticket=("avg_ticket", "mean"))
)
print(dow_summary.round(2).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(daily_df["date"], daily_df["revenue"], alpha=0.45, label="Daily Revenue")
axes[0].plot(daily_df["date"], daily_df["rolling_14d"], linewidth=2.0, label="14-Day Rolling Mean")
axes[0].axvline(pd.Timestamp(cfg.tablet_channel_start_date), linestyle="--", color="gray", label="Tablet Channel Audit Date")
axes[0].set_title("Daily Revenue Baseline & Trend")
axes[0].legend()

hourly_profile = hourly_df.groupby("hour_int", as_index=False)["volume"].mean()
colors = ["#d95f02" if h in cfg.peak_hours else "#1b9e77" for h in hourly_profile["hour_int"]]
axes[1].bar(hourly_profile["hour_int"], hourly_profile["volume"], color=colors)
axes[1].set_title("Mean Hourly Revenue (Peak Hours Highlighted)")
axes[1].set_xlabel("Hour of Day (06:00-21:00)")
plt.tight_layout()
plt.close(fig)

mix_summary = mix_df.groupby("category", as_index=False)["quantity_sold"].sum()
mix_summary["share_pct"] = (100.0 * mix_summary["quantity_sold"] / mix_summary["quantity_sold"].sum()).round(2)
print("\nProduct Mix Summary:")
print(mix_summary.to_string(index=False))